# 04 — Translation: mBART Vi↔Ede, Vi↔Bahnar (Kaggle / Colab / Local)

Fine-tunes `facebook/mbart-large-50` cho dịch:
- Vietnamese ↔ Ede
- Vietnamese ↔ Bahnar

**Checkpoint**: Tự động skip nếu model đã export.

> ⚠️ Corpus hiện tại là demo (8 cặp câu). Cần ít nhất 5k+ cặp câu cho production.

In [ ]:
# ── Cell 0: Environment + Paths (Kaggle / Colab / Local) ─────────────────────
import sys, os
from pathlib import Path

# Thêm notebook dir để import _env.py
if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')

from _env import *

CORPUS_DIR  = os.path.join(os.path.dirname(CHECKPOINT_DIR), 'corpora')
Path(CORPUS_DIR).mkdir(parents=True, exist_ok=True)
EXPORT_DIR  = os.path.join(WEIGHTS_DIR, 'translation')
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
TRANS_CKPT  = os.path.join(CHECKPOINT_DIR, 'translation')
Path(TRANS_CKPT).mkdir(parents=True, exist_ok=True)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ENV    : {ENV_NAME}')
print(f'Device : {device}')
print(f'CORPUS : {CORPUS_DIR}')
print(f'EXPORT : {EXPORT_DIR}')

# Checkpoint skip
CKPT_DONE = os.path.exists(os.path.join(EXPORT_DIR, 'config.json'))
if CKPT_DONE:
    print(f'\n✅ CHECKPOINT: Translation model đã export')
else:
    print('\n⏳ Chưa có translation model — sẽ train...')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhớ Save Version sau khi training xong!')

In [ ]:
# ── Cell 1: Cài dependencies ──────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.40.0', 'datasets>=2.18.0', 'accelerate>=0.28.0',
                'evaluate>=0.4.0', 'sacrebleu', 'sentencepiece', 'mlflow'],
               check=False)
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Build bilingual corpus ────────────────────────────────────────────
import json

ede_vi_pairs = [
    {'vi': 'Xin chào', 'ede': 'Khoa êa'},
    {'vi': 'Tôi đi học', 'ede': 'Kâo nao hoc'},
    {'vi': 'Cảm ơn bạn', 'ede': 'Bơng kâo lač'},
    {'vi': 'Nước uống ở đâu?', 'ede': 'Êa mnăm dôk ti řah?'},
    {'vi': 'Tôi không hiểu', 'ede': 'Kâo amâo thâo'},
    {'vi': 'Nhà bạn ở đâu?', 'ede': 'Sang ih dôk ti řah?'},
    {'vi': 'Tôi đói bụng', 'ede': 'Kâo ư hăng'},
    {'vi': 'Gọi cho tôi', 'ede': 'Khih kơ kâo'},
]

bah_vi_pairs = [
    {'vi': 'Xin chào', 'bah': 'Thơ pơtong'},
    {'vi': 'Tôi đi học', 'bah': 'Kơxai pơlar'},
    {'vi': 'Cảm ơn', 'bah': 'Môr kơxai'},
    {'vi': 'Nước', 'bah': 'Dak'},
    {'vi': 'Ngôi nhà', 'bah': 'Sang'},
    {'vi': 'Tôi không biết', 'bah': 'Kơxai thơh biêt'},
    {'vi': 'Bao nhiêu tiền?', 'bah': 'Bia do?'},
    {'vi': 'Đi thôi', 'bah': 'Pơlar bơh'},
]

with open(os.path.join(CORPUS_DIR, 'ede_vi.json'), 'w', encoding='utf-8') as f:
    json.dump(ede_vi_pairs, f, ensure_ascii=False, indent=2)
with open(os.path.join(CORPUS_DIR, 'bah_vi.json'), 'w', encoding='utf-8') as f:
    json.dump(bah_vi_pairs, f, ensure_ascii=False, indent=2)

print(f'Corpus saved: {os.listdir(CORPUS_DIR)}')
print(f'Ede pairs: {len(ede_vi_pairs)} | Bahnar pairs: {len(bah_vi_pairs)}')
print('⚠️  Demo corpus. Production cần 5k+ cặp câu.')

In [ ]:
# ── Cell 3: Load mBART-50 ──────────────────────────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip load — dùng model đã export')
else:
    from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

    MODEL_NAME = 'facebook/mbart-large-50'
    print(f'Loading {MODEL_NAME}...')
    tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
    model_mt  = MBartForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
    print(f'✓ mBART-50 loaded ({sum(p.numel() for p in model_mt.parameters())/1e6:.0f}M params)')

In [ ]:
# ── Cell 4: Training ───────────────────────────────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip training — model đã có')
else:
    from datasets import Dataset
    from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
    import evaluate

    sacrebleu = evaluate.load('sacrebleu')
    os.environ['MLFLOW_TRACKING_URI'] = os.path.join(LOGS_DIR, 'mlflow')
    os.makedirs(os.path.join(LOGS_DIR, 'mlflow'), exist_ok=True)

    all_src = [p['vi'] for p in ede_vi_pairs + bah_vi_pairs]
    all_tgt = [p.get('ede', p.get('bah', '')) for p in ede_vi_pairs + bah_vi_pairs]
    ds_combined = Dataset.from_dict({'src': all_src, 'tgt': all_tgt})
    MAX_LEN = 128

    def preprocess(batch):
        tokenizer.src_lang = 'vi_VN'
        model_inputs = tokenizer(batch['src'], max_length=MAX_LEN, truncation=True, padding='max_length')
        labels       = tokenizer(text_target=batch['tgt'], max_length=MAX_LEN, truncation=True, padding='max_length')
        model_inputs['labels'] = labels['input_ids']
        return model_inputs

    ds_proc = ds_combined.map(preprocess, batched=True, batch_size=16, num_proc=1)
    ds_proc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    split = ds_proc.train_test_split(test_size=0.2, seed=42)

    def compute_metrics(pred):
        preds  = pred.predictions
        labels = pred.label_ids
        labels[labels == -100] = tokenizer.pad_token_id
        decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        result = sacrebleu.compute(predictions=decoded_preds,
                                   references=[[l] for l in decoded_labels])
        return {'bleu': result['score']}

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_mt, padding=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=TRANS_CKPT,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-5,
        warmup_steps=20,
        num_train_epochs=10,
        fp16=(device == 'cuda'),
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='bleu',
        predict_with_generate=True,
        logging_steps=5,
        report_to=['mlflow'],
        push_to_hub=False,
        dataloader_num_workers=NUM_WORKERS,
    )

    trainer = Seq2SeqTrainer(
        model=model_mt, args=training_args,
        train_dataset=split['train'], eval_dataset=split['test'],
        tokenizer=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
    print(f'\n✅ mBART model saved → {EXPORT_DIR}')

    if IS_KAGGLE:
        print('⚠️  Kaggle: Nhấn Save Version để persist model!')

## ✅ mBART Translation Training Complete

| Platform | Model location |
|----------|---------------|
| Kaggle | `/kaggle/working/storage/weights/translation/` |
| Colab | `Drive/GenAI_ViMD/storage/weights/translation/` |

**Next**: Run `05_tts.ipynb`